In [3]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from tqdm import tqdm

# ==============================================================================
# 1. 全局参数 & H₂ 分子
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# Hilbert 空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
hi_ensemble = hi ** K

# 采样器 (增加 chains 和 sweep_size)
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_ensemble, rules=[single_rule]*K)
sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=32, sweep_size=32)


# =========================================================
# 3. 单态 Ansatz
# =========================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin, hidden=16, *, rngs):
        super().__init__()
        self.l1 = nnx.Linear(n_spin, hidden, rngs=rngs, param_dtype=complex)
        self.l2 = nnx.Linear(hidden, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        x = x.astype(complex)
        h = jnp.tanh(self.l1(x))
        return jnp.squeeze(self.l2(h))


# =========================================================
# 4. NES 总波函数（log Ψ）
# =========================================================
class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin, K):
        super().__init__()
        self.K = K

        key = jax.random.key(0)
        keys = jax.random.split(key, K)

        self.states = [
            SingleStateAnsatz(n_spin, rngs=nnx.Rngs(keys[i]))
            for i in range(K)
        ]

    def __call__(self, x_flat):
        n_spin = x_flat.shape[-1] // self.K
        x = x_flat.reshape(self.K, n_spin)

        # 构造 M
        M = []
        for i in range(self.K):
            row = [self.states[j](x[i]) for j in range(self.K)]
            M.append(jnp.array(row))
        M = jnp.stack(M)

        # 稳定 log det
        sign, logabs = jnp.linalg.slogdet(M + 1e-6*jnp.eye(self.K))
        return logabs + jnp.log(sign + 1e-12)


# =========================================================
# 5. 构建 MCState
# =========================================================
model = NESTotalAnsatz(n_spin=4, K=K)
vstate = nk.vqs.MCState(
    sampler,
    model,
    n_samples=2048
)

# Hack：塞入原哈密顿量
vstate.hamiltonian = ha


# =========================================================
# 6. NES Local Energy（核心）
# =========================================================
def local_energy_nes(vstate, samples):
    model = vstate.model
    ha = vstate.hamiltonian

    K = model.K
    n_spin = samples.shape[-1] // K

    def single_sample(x_flat):
        x = x_flat.reshape(K, n_spin)

        # --- M ---
        M = []
        for i in range(K):
            row = [model.states[j](x[i]) for j in range(K)]
            M.append(jnp.array(row))
        M = jnp.stack(M)

        # --- HΨ ---
        def Ham_psi(ansatz, x_i):
            xp, mels = ha.get_conn_padded(x_i)
            psi_vals = jax.vmap(ansatz)(xp)
            return jnp.sum(mels * psi_vals)

        H_mat = []
        for i in range(K):
            row = []
            for j in range(K):
                row.append(Ham_psi(model.states[j], x[i]))
            H_mat.append(row)
        H_mat = jnp.array(H_mat)

        # --- E_loc ---
        M_reg = M + 1e-4 * jnp.eye(K)
        el = jnp.linalg.solve(M_reg, H_mat)
        return jnp.trace(el).real

    return jax.vmap(single_sample)(samples)


# =========================================================
# 7. 自定义 Driver
# =========================================================
class NESVMC(nk.driver.AbstractVariationalDriver):
    def __init__(self, vstate, optimizer):
        super().__init__(vstate, optimizer)

    def _forward_and_backward(self):
        samples = self.state.samples

        def loss_fn(params):
            self.state.parameters = params
            e_loc = local_energy_nes(self.state, samples)
            return jnp.mean(e_loc)

        loss, grads = jax.value_and_grad(loss_fn)(self.state.parameters)

        self._loss = loss
        self._dp = grads


# =========================================================
# 8. 训练
# =========================================================
optimizer = nk.optimizer.Adam(learning_rate=1e-3)
driver = NESVMC(vstate, optimizer)

driver.run(n_iter=200)

# =========================================================
# 9. 提取能量（激发态）
# =========================================================
samples = vstate.samples.reshape(-1, K, 4)

def compute_M(model, samples):
    def single(x):
        M = []
        for i in range(K):
            row = [model.states[j](x[i]) for j in range(K)]
            M.append(jnp.array(row))
        return jnp.stack(M)

    Ms = jax.vmap(single)(samples)
    return Ms.mean(axis=0)

M = compute_M(model, samples)

energies = jnp.linalg.eigvalsh(M)
print("Estimated energies:", energies)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


TypeError: cannot reshape array of shape (32, 8) (size 256) into shape (2, 4) (size 8)